In [1]:

import deeplabcut
import os
from deeplabcut.modelzoo import build_weight_init
import shutil
from modules.dlc_utils import set_transform_prob

project_path = 'projects/rat2'
config_path = os.path.join(project_path, "config.yaml")
project_config = deeplabcut.auxiliaryfunctions.read_config(config_path)

Loading DLC 3.0.0rc13...


c:\Users\jiefei\anaconda3\envs\DEEPLABCUT\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# 1253 frames: Trained with superanimal_quadruped rtmpose_s + top-down detector + unfreeze bn stats + default detector training epochs
shuffle = 1

# initialize model weight

In [3]:
# superanimal_name = 'superanimal_mouse'
superanimal_name = 'superanimal_quadruped'
model_name = "rtmpose_s"
# detector_name="fasterrcnn_resnet50_fpn_v2"
detector_name = None
weight_init = build_weight_init(
            cfg = config_path,
            super_animal= superanimal_name,
            model_name=model_name,
            detector_name=detector_name,
            with_decoder=False
)


# Create training data

In [4]:
## delete `training-datasets` folder
path1 = os.path.join(project_path, 'training-datasets/iteration-0/UnaugmentedDataSet_Sleap_Rat_testOct2')
if os.path.exists(path1):
    name_contain = f"shuffle{shuffle}"
    # delete everything that contains `shuffle{shuffle}` in the name
    for item in os.listdir(path1):
        if name_contain in item:
            os.remove(os.path.join(path1, item))

path2 = os.path.join(project_path, f'dlc-models-pytorch/iteration-0/Sleap_Rat_testOct2-trainset95shuffle{shuffle}')
if os.path.exists(path2):
    shutil.rmtree(path2)
    
dt = deeplabcut.create_training_dataset(
    config_path, 
    Shuffles=[shuffle],    
    weight_init=weight_init, 
    net_type=model_name,  
    detector_type=detector_name,
    userfeedback=False)

F:\code\pose_track\projects\rat2\labeled-data\RAT 11 FR1\CollectedData_rats.h5  not found (perhaps not annotated).


# replace data augmentation parameters

In [5]:
from deeplabcut.core.config import read_config_as_dict
import deeplabcut.pose_estimation_pytorch as dlc_torch
import yaml

loader = dlc_torch.DLCLoader(
    config=config_path,  
    trainset_index=0,
    shuffle=shuffle,
)

# Get the pytorch config
pytorch_config_path = loader.model_folder / "pytorch_config.yaml"
model_cfg = read_config_as_dict(pytorch_config_path)

In [6]:
# Set freeze_bn_stats=False for GPU training with large batch size
model_cfg["detector"]["model"]["freeze_bn_stats"] = False
# model_cfg["detector"]["train_settings"]["batch_size"] = 4

dlc_torch.config.write_config(pytorch_config_path, model_cfg)

# Add horizontal flip augmentation

In [7]:
{i: project_config['bodyparts'][i] for i in range(len(project_config['bodyparts']))}

{0: 'head',
 1: 'nose',
 2: 'spine1',
 3: 'spine2',
 4: 'spine3',
 5: 'tailbase',
 6: 'tail1',
 7: 'tail2',
 8: 'tail_tip',
 9: 'L_hip',
 10: 'L_backpaw',
 11: 'R_backpaw',
 12: 'L_shoulder',
 13: 'R_frontpaw',
 14: 'R_shoulder',
 15: 'R_hip',
 16: 'R_knee',
 17: 'L_knee',
 18: 'L_frontpaw'}

In [8]:
# Symmetric pairs (left-right): 
# L_hip(9) ↔ R_hip(15), L_backpaw(10) ↔ R_backpaw(11), 
# L_shoulder(12) ↔ R_shoulder(14), L_frontpaw(18) ↔ R_frontpaw(13), 
# L_knee(17) ↔ R_knee(16)

model_cfg["data"]["train"]["hflip"] = {
    "p": 0.5,  # 50% probability
    "symmetries": [[9, 15], [10, 11], [12, 14], [13, 18], [16, 17]]
}

dlc_torch.config.write_config(pytorch_config_path, model_cfg)
print(model_cfg["data"]["train"]["hflip"])

{'p': 0.5, 'symmetries': [[9, 15], [10, 11], [12, 14], [13, 18], [16, 17]]}


# Train

In [9]:
# delete all pt files 
import glob
pt_files = glob.glob(f'projects/rat_pose/dlc-models-pytorch/iteration-0/Sleap_Rat_testOct2-trainset95shuffle{shuffle}/train/*.pt')
for f in pt_files:
    os.remove(f)

In [10]:
deeplabcut.train_network(
    config_path,
    shuffle=shuffle,
    epochs=100,
    save_epochs=10,
    detector_epochs = 50,
    superanimal_name=superanimal_name,
    batch_size= 16,
    keepdeconvweights=False,
    device="cuda:0",
    superanimal_transfer_learning=True
    )

Training with configuration:
data:
  bbox_margin: 20
  colormode: RGB
  inference:
    normalize_images: True
    top_down_crop:
      width: 256
      height: 256
  train:
    affine:
      p: 0.5
      rotation: 30
      scaling: [1.0, 1.0]
      translation: 0
    gaussian_noise: 12.75
    motion_blur: True
    normalize_images: True
    top_down_crop:
      width: 256
      height: 256
    random_bbox_transform:
      shift_factor: 0.16
      shift_prob: 0.3
      scale_factor: [0.75, 1.25]
      scale_prob: 1.0
      p: 1.0
    hflip:
      p: 0.5
      symmetries: [[9, 15], [10, 11], [12, 14], [13, 18], [16, 17]]
detector:
  data:
    colormode: RGB
    inference:
      normalize_images: True
    train:
      affine:
        p: 0.5
        rotation: 30
        scaling: [1.0, 1.0]
        translation: 40
      collate:
        type: ResizeFromDataSizeCollate
        min_scale: 0.4
        max_scale: 1.0
        min_short_side: 128
        max_short_side: 1152
        multiple_of: 